# GeoTIGER: local geocoding demo with Durham crime data

This notebook downloads and caches the public Durham Police Department crime table, prepares Durham TIGER/Line address ranges, and geocodes the full crime table locally. The public crime addresses are obfuscated to the nearest 100 block, so the demo changes `*00` addresses to `*50` before geocoding.

The range interpolation uses North Carolina State Plane EPSG:2264, not latitude/longitude. The prepared table keeps WGS84 latitude/longitude columns for mapping. The public Durham crime table and TIGER ranges are downloaded only when their local cache files are missing; the geocoding run itself is local.

In [ ]:
import time
from pathlib import Path

import geopandas as gpd
import pandas as pd
from IPython.display import display

from geotiger import (
    Geocoder,
    GeocoderConfig,
    GeoTIGERStore,
    InterpolationConfig,
    load_durham_crime,
    make_durham_inputs,
    prepare_ranges,
    state_plane_crs,
)
from geotiger.sources import download_tiger_ranges
from geotiger.viz import matches_map

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
CACHE_DIR = REPO_ROOT / 'data' / 'durham_demo'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR = REPO_ROOT / 'notebooks' / 'assets'
CRIME_CACHE = (ASSET_DIR / 'durham_crime.csv.gz') if (ASSET_DIR / 'durham_crime.csv.gz').exists() else (CACHE_DIR / 'durham_crime.csv.gz')
RANGES_CACHE = (ASSET_DIR / 'durham_tiger_ranges.parquet') if (ASSET_DIR / 'durham_tiger_ranges.parquet').exists() else (CACHE_DIR / 'durham_tiger_ranges.parquet')
PREPARED_CACHE = CACHE_DIR / 'durham_prepared_addresses.parquet'
DATABASE = CACHE_DIR / 'durham_addresses.duckdb'
RESULT_CACHE = CACHE_DIR / 'durham_geocoded.parquet'

print('Cache directory:', CACHE_DIR)
print('Default NC interpolation CRS:', state_plane_crs('NC'))

## 1. Load the large public crime table

`load_durham_crime` paginates the City of Durham ArcGIS table once and stores it as a gzip-compressed CSV. Re-running this cell uses the local cache and does not make a request.

In [ ]:
crime_raw = load_durham_crime(CRIME_CACHE)
print(f'{len(crime_raw):,} public crime records')
print('Compressed cache:', f'{CRIME_CACHE.stat().st_size / 1_000_000:.1f} MB')
display(crime_raw[['INCI_ID', 'DATE_REPT', 'REPORTEDAS', 'ADDRESS2']].head())

## 2. Move public `*00` addresses to the `*50` block midpoint

The source table intentionally reports addresses at the nearest 100-block level. For example, `1400 SNOWCREST TRL` becomes `1450 SNOWCREST TRL`. This is an analytic midpoint, not a recovery of the original incident location.

In [ ]:
crime = make_durham_inputs(crime_raw)
display(crime[['ADDRESS2', 'address', 'city', 'state']].head(10))
print('Addresses changed to block midpoints:', (crime['ADDRESS2'] != crime['address']).sum())

## 3. Download and cache Durham TIGER/Line ranges

This uses the Census TIGER/Line address-range table for Durham County, NC. The `pygris` source has ZIP fields but no textual city field, so the later geocoder uses strict state blocking and local street/house-number blocking; city is retained on the crime inputs for parsing and auditability.

In [ ]:
if RANGES_CACHE.exists():
    ranges = gpd.read_parquet(RANGES_CACHE)
else:
    ranges = download_tiger_ranges('NC', county='Durham', year=2024, cache=True)
    ranges.to_parquet(RANGES_CACHE, index=False)
print(f'{len(ranges):,} TIGER range segments')
print('Input CRS:', ranges.crs)
display(ranges[['FULLNAME', 'LFROMHN', 'LTOHN', 'RFROMHN', 'RTOHN', 'ZIPL', 'ZIPR']].head())

## 4. Prepare the local interpolated address table

Because crime locations are already block midpoints, both offsets are set to zero. The interpolation is still done in EPSG:2264 state-plane feet; only the saved `latitude`/`longitude` columns are WGS84. The resulting prepared table can be reused for future local runs.

In [ ]:
range_config = InterpolationConfig(
    projected_crs=state_plane_crs('NC'),
    end_offset_m=0,
    side_offset_m=0,
)
if PREPARED_CACHE.exists():
    prepared = pd.read_parquet(PREPARED_CACHE)
else:
    prep_started = time.perf_counter()
    prepared = prepare_ranges(
        ranges,
        state='NC',
        config=range_config,
        source='durham-tiger-2024',
    )
    prepared.to_parquet(PREPARED_CACHE, index=False)
    print(f'Preparation seconds: {time.perf_counter() - prep_started:.2f}')
print(f'{len(prepared):,} expanded address-side points')
print('Interpolation CRS:', prepared['interpolation_crs'].unique())
display(prepared[['house_number', 'street_norm', 'side', 'latitude', 'longitude', 'interpolation_crs']].head())

## 5. Persist the prepared table in DuckDB

This database is local and reusable. If it already exists, the notebook opens it instead of rebuilding it.

In [ ]:
store = GeoTIGERStore(DATABASE)
store.create()
if store.count() == 0:
    store.ingest_candidates(prepared, replace=True)
    store.set_metadata(
        source='durham-tiger-2024',
        interpolation_crs=state_plane_crs('NC'),
        end_offset_m=0,
        side_offset_m=0,
    )
print(f'{store.count():,} rows in {DATABASE}')
print(store.metadata())

## 6. Geocode all crime records locally

`strict_locality=False` is intentional here because the raw TIGER range table has no city-name column and many segments have no ZIP. State is still required, and the street-prefix plus house-number blocks keep the candidate search local. The result includes the best match, score/status, coordinates, and every blocked candidate.

In [ ]:
geocoder = Geocoder(
    store,
    config=GeocoderConfig(
        strict_locality=False,
        house_number_tolerance=25,
        street_fallback=False,
        auto_match_threshold=90,
    ),
)
result = geocoder.geocode(
    crime,
    address_column='address',
    city_column='city',
    state_column='state',
    zip_column=None,
)
print(result.timings.to_dict())
display(result.matches[['INCI_ID', 'address', 'score', 'match_status', 'match_latitude', 'match_longitude']].head())

In [ ]:
result.matches.to_parquet(RESULT_CACHE, index=False)
print(f'Saved geocoded results to {RESULT_CACHE}')
display(result.matches['match_status'].value_counts(dropna=False).rename('records'))
print(f'Potential candidates returned: {len(result.candidates):,}')

## 7. Offline result map

The default map has no basemap tile URL, so the result can be opened without sending coordinates to a tile provider. For notebook responsiveness, the display samples up to 5,000 automatically matched points; the full geocoded table remains in RESULT_CACHE.

In [ ]:
map_rows = result.matches.loc[result.matches['auto_assigned']].sample(
    n=min(5_000, int(result.matches['auto_assigned'].sum())),
    random_state=42,
)
result_map = matches_map(map_rows, tiles=None, zoom_start=12)
result_map

### Interpretation

The geocoded point is the midpoint of the reported 100 block, not the exact incident location. The compressed source cache, TIGER range cache, prepared DuckDB database, and geocoded output are written under `data/durham_demo/`, which is ignored by Git by default. Delete that directory to force a fresh download/rebuild.